# **Medical Information Extraction with OpenAI API**

Author: **Rai Peladas** <br>
Date Created: **February 27, 2026** <br>
Date Modified: **March 9, 2026** <br>
Version: **2** <br>

This project explores the use of the OpenAI API for automated medical information extraction from unstructured patient transcripts. In many healthcare settings, medical professionals write down consultations in the natural language. They capture details such as symptoms, diagnoses, and treatments in it. While these transcripts are useful for recordkeeping, their free-form nature makes manual extraction time-consuming and prone to error. 

The model created for this particular issue is built to automatically extract patient information and map it to the appropriate treatments and corresponding ICD-10 codes. Using prompt engineering, function definition, and Pandas-based data processing, the system is designed to return structured outputs containing the patient's age, recommended treatment, and corresponding ICD-10 classification. This all in all makes the results easier to integerate into other healthcare and insurance processes. 

<p align="center">
  <img src="maomao_in_the_house.jpg" alt="Maomao and Dr. House" width="500">
</p>

## **Key Techniques Used**
- **Prompt Engineering** to guide the model towards a more consistent information extraction
- **Function Definition/ Tool Schema Design** to contrain outputs into a predictable format such as the age, treatment, and ICD-10 code
- **System Prompting** to establish the model's role and extraction behavior
- **Pandas Data Modification** to process transcripts row by row and organize results into a structured dataset
- **JSON Output Handling** to make the extracted information easier to store and validate

## **About the Dataset**
The dataset contains anonymized medical transcriptions categorized by specialty.

### **transcriptions.csv**
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

## **Model Development**

In [62]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

In [63]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   medical_specialty  5 non-null      object
 1   transcription      5 non-null      object
dtypes: object(2)
memory usage: 208.0+ bytes


In [64]:
# Initialize the OpenAI client
client = OpenAI()

In [65]:
# Define function definition
function_definition = [{
    'type': 'function',
    'function': {
        'name': 'get_recommended_treatment',
        'description': "Extract the numeric age of the patient per row from the 'transcription' column if available. Return null if not found. Then, derive recommended treatment for each patient per column given the transcription details from the transcription column. Match each recommended treatment with the corresponding International Classification of Diseases (ICD) code. Return the ICD-10 code in standard format (e.g., A00.0). Return null if uncertain.", 
        'parameters': {
            'type': 'object',
            'properties': {
                'age': {'type': ['string', 'null'],
                       'description': 'Patient age'},
                'recommended_treatment': {
                    'type': ['string', 'null'],
                    'description': 'Recommended treatment for the patient given the transcription details.'
                },
                'icd_code': {
                    'type': ['string', 'null'],
                    'description': 'Matched International Classification of Diseases (ICD) code with the provided recommended treatment.'
                }
            }        
        }
    }
}]

In [66]:
# Define system instructions
messages = [
    {
        'role': 'system',
        'content': 'You are data engineer specializing in the medical data. For each transcription, extract age, recommended treatment, and ICD-10 code. Return structured JSON for each entry.'
    }
]

In [67]:
# Append each row from df as separate user message 
for idx, row in df.iterrows():
    messages.append({
        'role': 'user',
        'content': (
            f"Medical specialty: {row['medical_specialty']}\n"
            f"Transcription: {row['transcription']}"
        )
    })

In [68]:
# Call chat completions endpoint 
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=function_definition
)

In [69]:
# Concatenate medical specialty entries from df to new data list
new_data = []
for i in range(len(df)):
    data = json.loads(response.choices[0].message.tool_calls[i].function.arguments)
    data['medical_specialty'] = df.loc[i, 'medical_specialty']
    new_data.append(data)

In [70]:
# Convert new data list into new dataframe 
df_structured = pd.DataFrame(new_data)
df_structured.head()

,age,recommended_treatment,icd_code,medical_specialty
0,23,Try Zyrtec instead of Allegra and samples of N...,J30.9,Allergy / Immunology
1,41,Operative fixation for Achilles tendon rupture,S86.011,Orthopedic
2,30,Laparoscopic Roux-en-Y gastric bypass,E66.01,Bariatrics
3,50,Tracheostomy and bronchoscopy for airway obstr...,J95.850,Cardiovascular / Pulmonary
4,66,Self-catheterization as needed and Flomax for BPH,N40.0,Urology


## **Conclusion**

This project demonstrates that the use of proper data handling, prompting, and function schemas with the OpenAI API can extract structured information such as the age, potential treatment, and ICD-10 code from unstructured data coming from transcripts created by medical professionals written in natural language. In turn, this process creates a structured dataset containing consistent information across varying entries from the transcripts provided.

The system prompt establishes expectations for the LLM to operate within the proper context, positioning itself as a data engineer responsible for structured data extraction. This provides the model with the relevant framing for how the information should be interpreted and processed. In this case, the extraction was primarily guided by the function schema constraints, which clearly specify the information to be extracted such as the age, treatment, and corresponding ICD-10 code. These constraints serve as both direction and guardrails, ensuring that the model returns structured outputs and defaults to null when the required information cannot be confidently identified. This reduces hallucinations and clarifies the expectations placed on the model. Additionally, the extraction was implemented through a single function call to reduce recurring API callbacks that could affect both processing speed and financial cost. Since the output is returned as JSON, Pandas was then used to structure the results into a dataset containing the final organized information, making it ready for scalable processing across other potential healthcare and insurance workflows.

However, there are still shortcomings in this type of approach. The mapping of recommended treatments to ICD-10 codes may remain approximate, and ambiguous diagnoses within transcripts can make extraction dependent on prompt interpretation. In such cases, the model may return null when uncertainty is detected, which acts as a safeguard but may require additional downstream handling.

For further improvements, rule-based checks can be introduced to validate ICD-10 codes by referencing an external database through retrieval methods such as RAG. Deterministic parsing techniques, such as using regular expressions to extract age, can also complement the LLM workflow. Finally, incorporating a human-in-the-loop process would allow extraction accuracy to be evaluated against verified ground truth data, supporting continuous refinement and improved reliability of the system.